To introduce our data to the user we decided to present the cohort in simple pie charts. First the focus is on how many people actually use social media, what kind is the most popular and how long do the users spend there daily. Pie charts are a great option here as they are easily readable and get a good first idea of data.
Next, we show bar chart with counts of cohort's occupation, and grouped marital status and gender, (maybe should group age by accupation)
The summary of the questions that they were asked can also be looked at before the user dives deeper in the analysis.

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import numpy as np
from scipy.stats import gaussian_kde
data = pd.read_csv('../data/smmh.csv')

In [34]:
"""
export_for_html.py
──────────────────
Run this in your notebook AFTER `data` is already loaded.
It computes every value needed by grandma_tab1.html and
writes  survey_data.json  next to this file.

Usage (in notebook cell):
    exec(open("export_for_html.py").read())
"""

import json, re
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# ── 0. Gender normalisation ──────────────────────────────────────────────────
data['Gender_Grouped'] = data['2. Gender'].replace({
    'Nonbinary ':          'Non-binary',
    'Non-binary':          'Non-binary',
    'NB':                  'Non-binary',
    'Non binary ':         'Non-binary',
    'Male':                'Male',
    'Female':              'Female',
    'unsure ':             'Other',
    'Trans':               'Other',
    'There are others???': 'Other',
})

# ── 1. Column rename ─────────────────────────────────────────────────────────
SHORT = {
    "1. What is your age?":                                    "age",
    "Gender_Grouped":                                          "gender",
    "3. Relationship Status":                                  "relationship",
    "4. Occupation Status":                                    "occupation",
    "5. What type of organizations are you affiliated with?":  "org",
    "6. Do you use social media?":                             "uses_sm",
    "7. What social media platforms do you commonly use?":     "platforms",
    "8. What is the average time you spend on social media every day?": "daily_time",
    "9. How often do you find yourself using Social media without a specific purpose?":  "q9",
    "10. How often do you get distracted by Social media when you are busy doing something?": "q10",
    "11. Do you feel restless if you haven't used Social media in a while?":             "q11",
    "12. On a scale of 1 to 5, how easily distracted are you?":                         "q12",
    "13. On a scale of 1 to 5, how much are you bothered by worries?":                  "q13",
    "14. Do you find it difficult to concentrate on things?":                            "q14",
    "15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?": "q15",
    "16. Following the previous question, how do you feel about these comparisons, generally speaking?": "q16",
    "17. How often do you look to seek validation from features of social media?":       "q17",
    "18. How often do you feel depressed or down?":                                      "q18",
    "19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?": "q19",
    "20. On a scale of 1 to 5, how often do you face issues regarding sleep?":           "q20",
}
df = data.rename(columns={k: v for k, v in SHORT.items() if k in data.columns})

out = {}

# ── 2. Top-level stats ───────────────────────────────────────────────────────
out["total"]    = int(len(df))
out["users"]    = int((df["uses_sm"] == "Yes").sum())  if "uses_sm" in df.columns else None
out["nonUsers"] = int((df["uses_sm"] == "No").sum())   if "uses_sm" in df.columns else None
out["avgAge"]   = round(float(df["age"].dropna().mean()), 1) if "age" in df.columns else None

# ── 3. Users vs Non-users pie ────────────────────────────────────────────────
if "uses_sm" in df.columns:
    vc = df["uses_sm"].value_counts()
    out["usersPie"] = [{"name": k, "value": int(v)} for k, v in vc.items()]

# ── 4. Platforms pie ─────────────────────────────────────────────────────────
PLATFORM_LIST = [
    "YouTube","Instagram","Facebook","Twitter","TikTok",
    "Snapchat","Discord","Pinterest","Reddit","LinkedIn",
]
if "platforms" in df.columns:
    counts_p = {}
    for p in PLATFORM_LIST:
        n = int(df["platforms"].dropna().str.contains(p, case=False, na=False).sum())
        if n: counts_p[p] = n
    out["platforms"] = sorted(
        [{"name": k, "value": v} for k, v in counts_p.items()],
        key=lambda x: -x["value"]
    )

# ── 5. Age distribution bars ─────────────────────────────────────────────────
if "age" in df.columns:
    ages = df["age"].dropna()

    def age_label(a):
        if a < 15:  return "<15"
        if a <= 30: return str(int(a))
        if a <= 40: return "31-40"
        return "40+"

    age_series = ages.apply(age_label)
    order = [str(i) for i in range(15, 31)] + ["31-40", "40+", "<15"]
    vc = age_series.value_counts()
    out["ageBins"] = [
        {"label": lbl, "n": int(vc.get(lbl, 0))}
        for lbl in order if vc.get(lbl, 0) > 0
    ]

    # KDE curves
    if "uses_sm" in df.columns:
        age_u  = df[df["uses_sm"] == "Yes"]["age"].dropna().values
        age_nu = df[df["uses_sm"] == "No"]["age"].dropna().values
        x_range = np.linspace(float(ages.min()), float(ages.max()), 80)

        def kde_points(arr, x):
            if len(arr) < 2: return []
            kde = gaussian_kde(arr)
            return [{"x": round(float(xi), 2), "y": round(float(yi), 6)}
                    for xi, yi in zip(x, kde(x))]

        out["ageKde"] = {
            "users":    kde_points(age_u,  x_range),
            "nonUsers": kde_points(age_nu, x_range),
        }

# ── 6. Gender × Relationship ─────────────────────────────────────────────────
if "gender" in df.columns and "relationship" in df.columns:
    preferred_rel = ["Single", "In a relationship", "Married", "Divorced"]
    cross = df.groupby(["relationship", "gender"]).size().unstack(fill_value=0)
    rel_order = [r for r in preferred_rel if r in cross.index] + \
                [r for r in cross.index  if r not in preferred_rel]
    cross = cross.reindex(rel_order)

    out["relationship"] = []
    for rel in rel_order:
        row = {"status": rel, "total": int(cross.loc[rel].sum()), "breakdown": {}}
        for g in cross.columns:
            row["breakdown"][g] = int(cross.loc[rel, g])
        out["relationship"].append(row)

    out["genders"] = list(cross.columns)

# ── 7. Occupation ─────────────────────────────────────────────────────────────
if "occupation" in df.columns:
    vc = df["occupation"].value_counts()
    out["occupation"] = [{"label": k, "n": int(v)} for k, v in vc.items()]

# ── 8. Daily screen time ─────────────────────────────────────────────────────
TIME_ORDER = [
    "Less than an Hour","Between 1 and 2 hours","Between 2 and 3 hours",
    "Between 3 and 4 hours","Between 4 and 5 hours","More than 5 hours",
]
if "daily_time" in df.columns:
    vc = df["daily_time"].value_counts()
    out["dailyTime"] = [
        {"label": lbl, "n": int(vc.get(lbl, 0))}
        for lbl in TIME_ORDER if vc.get(lbl, 0) > 0
    ]

# ── 9. Mental health averages ─────────────────────────────────────────────────
MH_COLS = ["q9","q10","q11","q12","q13","q14","q15","q17","q18","q19","q20"]
mh_avgs = {}
for col in MH_COLS:
    if col in df.columns:
        numeric = pd.to_numeric(df[col], errors="coerce")
        mh_avgs[col] = round(float(numeric.mean()), 2) if numeric.notna().any() else None
out["mhAverages"] = mh_avgs

# ── 10. Write JSON ────────────────────────────────────────────────────────────
OUTPUT_PATH = "survey_data.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

print(f"✓ Exported {len(out)} sections → {OUTPUT_PATH}")
print(f"  Total respondents : {out['total']}")
print(f"  SM users          : {out['users']} ({out['users']/out['total']*100:.1f}%)")
print(f"  Avg age           : {out['avgAge']}")
print(f"  Relationship rows : {len(out.get('relationship', []))}")
print(f"  Platform rows     : {len(out.get('platforms', []))}")
print(f"  Age bins          : {len(out.get('ageBins', []))}")

✓ Exported 13 sections → survey_data.json
  Total respondents : 481
  SM users          : 478 (99.4%)
  Avg age           : 26.1
  Relationship rows : 4
  Platform rows     : 9
  Age bins          : 19
